# 🏥 Facial Skin Disease Classification: Modular Production Pipeline
### Clean Architecture · Transfer Learning · Scalable Training & Deployment
---

This notebook demonstrates the end-to-end execution of the modularized research and production pipeline. Rather than maintaining monolithic script files, the system is organized into decoupled modules under `src/` and `configs/`:

```
Facial_Skin_Diesease_Synthetic_Generation/
├── configs/
│   └── config.py              # Centralized experiment hyperparameters & directories
├── src/
│   ├── data/
│   │   ├── dataset.py         # Leakage-free dataset ingestion, validation & split
│   │   ├── pipeline.py        # High-performance tf.data input pipelines & augmentations
│   │   └── utils.py           # Class weighting, label maps & serialization
│   ├── models/
│   │   ├── backbones.py       # Pretrained backbones (EfficientNet, ResNet, DenseNet, etc.)
│   │   ├── heads.py           # Standardized classification head & custom layers
│   │   └── losses.py          # Class-weighted multi-class Focal Loss
│   ├── training/
│   │   ├── callbacks.py       # Checkpointing, early stopping & LR decay
│   │   └── trainer.py         # Progressive 3-phase training engine
│   ├── evaluation/
│   │   ├── metrics.py         # Comprehensive clinical evaluation suite
│   │   ├── visualization.py   # Confusion matrices & training trajectory curves
│   │   └── ensemble.py        # Macro-F1 weighted soft voting ensemble
│   └── inference/
│       └── predictor.py       # Single-image & batch production inference engine
└── Notebooks/
    ├── Notebook.ipynb         # Full monolithic experimental benchmark
    └── Modular_Pipeline.ipynb # Modular pipeline execution tutorial
```

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
from configs.config import default_config
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

## 1. Dataset Discovery, Quality Auditing & Leakage-Free Splitting
We discover the raw image dataset, sanitize corrupted files, perform a strict **Stratified 70/15/15 Split**, and apply bounded resampling strictly to the training fold to eliminate any risk of data leakage.

In [ ]:
from src.data.dataset import discover_dataset, clean_dataset, split_dataset, balance_dataset
from src.data.pipeline import make_dataset
from src.data.utils import compute_class_weights, save_label_map
from src.evaluation.visualization import plot_class_distribution, plot_sample_images

config = default_config
df_full, class_names = discover_dataset(config.data_dir)
df_clean = clean_dataset(df_full)
train_df_raw, val_df, test_df = split_dataset(df_clean, test_size=config.test_size, seed=config.seed)

train_target = int(round(train_df_raw['class'].value_counts().mean()))
df_balanced = balance_dataset(train_df_raw, target=train_target, max_cap=train_target, seed=config.seed)
plot_class_distribution(train_df_raw, df_balanced, config.plot_dir)

save_label_map(class_names, config.save_dir / 'label_map.json')
class_weights = compute_class_weights(df_balanced['label'].values, len(class_names))
print(f'Class weights: {class_weights}')

## 2. High-Performance Input Pipelines (`tf.data`)
Construct autotuned, prefetched `tf.data.Dataset` streams with clinical data augmentations for training, and deterministic evaluation streams for validation and testing.

In [ ]:
train_ds = make_dataset(df_balanced['path'].values, df_balanced['label'].values, img_size=config.img_size, batch_size=config.batch_size, augment=True, shuffle=True)
val_ds = make_dataset(val_df['path'].values, val_df['label'].values, img_size=config.img_size, batch_size=config.batch_size, augment=False, shuffle=False)
test_ds = make_dataset(test_df['path'].values, test_df['label'].values, img_size=config.img_size, batch_size=config.batch_size, augment=False, shuffle=False)

plot_sample_images(train_ds, class_names, config.plot_dir)

## 3. Phased Transfer Learning & Progressive Unfreezing
Instantiate the selected neural backbone and execute the **3-Phase Progressive Unfreezing Protocol** (Head Warmup $\to$ Partial Top-Layer Unfreezing $\to$ Deep Fine-Tuning) using `PhasedTrainer`.

In [ ]:
from src.models.backbones import get_backbone
from src.training.trainer import PhasedTrainer
from src.evaluation.metrics import evaluate_model
from src.evaluation.visualization import plot_training_history, plot_confusion_matrix, plot_roc_curves, plot_per_class_metrics

trainer = PhasedTrainer(config)
model_name = 'EfficientNetV2B0'
base_model = get_backbone(model_name, img_size=config.img_size)
model, histories, _ = trainer.train_model(
    model_name=model_name,
    base_model=base_model,
    unfreeze_phase2=100,
    unfreeze_phase3=200,
    train_ds=train_ds,
    val_ds=val_ds,
    class_weight_dict=class_weights,
)

metrics = evaluate_model(model, test_ds, class_names, model_name)
plot_training_history(histories, model_name, config.plot_dir)
plot_confusion_matrix(metrics['y_true'], metrics['y_pred'], class_names, model_name, config.plot_dir)
plot_roc_curves(metrics['y_true'], metrics['y_pred_prob'], class_names, model_name, config.plot_dir)
plot_per_class_metrics(metrics['report'], class_names, model_name, config.plot_dir)

## 4. Multi-Model Performance-Weighted Soft Voting Ensemble
Load all trained model checkpoints from `saved_models/` and compute validation Macro-F1 weighted probability predictions across the held-out test split.

In [ ]:
from src.evaluation.ensemble import WeightedEnsemble

ensemble = WeightedEnsemble(save_dir=config.save_dir, plot_dir=config.plot_dir, class_names=class_names)
n_models = ensemble.load_models()
print(f'Loaded {n_models} models.')
if n_models >= 2:
    ens_results = ensemble.evaluate_ensemble(test_ds)

## 5. Production Inference & Standalone Diagnostic Predictions
Demonstrate real-world clinical inference on individual patient images using the production-ready `DiseasePredictor` class.

In [ ]:
from src.inference.predictor import DiseasePredictor

model_files = sorted(config.save_dir.glob('*_final.keras'))
if model_files:
    predictor = DiseasePredictor(model_path=model_files[0], label_map=config.save_dir / 'label_map.json')
    pred_class, conf, probs = predictor.predict_image(test_df.iloc[0]['path'])
    print(f'Predicted: {pred_class} ({conf:.2%})')
    predictor.plot_prediction(test_df.iloc[0]['path'], pred_class, conf, probs, model_name=model_files[0].stem)